In [ ]:
%pip install ipywidgets httpx python-dotenv pydantic jose nest_asyncio

In [1]:
import asyncio
import os
import getpass
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()

from hinge_client import HingeClient, HingeAuthError, HingeEmail2FAError
from hinge_models import PhotoContent, AnswerContent, PromptContent

print(
    "✅ Setup complete. You can now proceed to the Authentication cell."
)

✅ Setup complete. You can now proceed to the Authentication cell.


In [2]:
load_dotenv()
phone_number = os.getenv(
    "HINGE_PHONE_NUMBER"
)

if not phone_number:
    phone_number = input(
        "Enter your phone number (e.g., +11234567890): "
    )

print(
    f"Attempting to log in with phone number: {phone_number}..."
)

client = HingeClient(
    phone_number=phone_number
)
loop = asyncio.get_event_loop()


async def authenticate():
    global client
    try:
        if not await client.is_session_valid():
            print(
                "Session not valid or expired. Starting new login..."
            )
            await client.initiate_login()
            otp = getpass.getpass(
                "Enter the OTP you received via SMS: "
            )
            try:
                await client.submit_otp(
                    otp.strip()
                )
            except HingeEmail2FAError as e:
                print(
                    f"Email 2FA is required. A code was sent to {e.email}."
                )
                email_code = getpass.getpass(
                    "Enter the code from your email: "
                )
                await client.submit_email_code(
                    email_code.strip(),
                    e.case_id
                )

        print(
            "✅ Authentication successful!"
        )
        return True

    except HingeAuthError as e:
        print(
            f"❌ Authentication Error: {e}"
        )
        return False
    except Exception as e:
        print(
            f"❌ An unexpected error occurred: {e}"
        )
        return False


auth_success = loop.run_until_complete(
    authenticate()
)


Attempting to log in with phone number: +4917623197452...
2025-09-03T20:28:36.041502Z [info     ] Using existing session data    filename=hinge_client.py func_name=__init__ lineno=106 phone_number=+4917623197452
2025-09-03T20:28:36.043073Z [info     ] Loaded 93 recommendations from file. filename=hinge_client.py func_name=_load_recommendations lineno=985
2025-09-03T20:28:36.080012Z [info     ] Loaded prompts from cache      filename=hinge_client.py func_name=_load_prompts_from_cache lineno=1012 total_categories=14 total_prompts=419
2025-09-03T20:28:36.080653Z [info     ] Session validity check         filename=hinge_client.py func_name=is_session_valid hinge_token_valid=True is_valid=True lineno=1148 sendbird_token_valid=True
✅ Authentication successful!


In [3]:
fetch_button = widgets.Button(
    description="Fetch Recommendations",
    button_style='primary'
)
recycle_button = widgets.Button(
    description="Recycle Profiles",
    button_style='info'
)
like_limit_label = widgets.Label()
main_output = widgets.Output()

recommendations_data = {}


async def update_like_limit_display():
    """Fetch and update the like limit label."""
    try:
        limit = await client.get_like_limit()
        like_limit_label.value = f"❤️ Likes Left: {limit.likes_left} | ⭐ Superlikes Left: {limit.super_likes_left}"
    except Exception as e:
        like_limit_label.value = "Could not fetch like limit."


async def display_profiles():
    """Fetch full content and render interactive widgets for each profile."""
    global recommendations_data
    with main_output:
        clear_output(
            wait=True
        )
        if not recommendations_data:
            print(
                "No recommendations to display. Try fetching or recycling."
            )
            return

        print(
            "Fetching full profile details..."
        )
        all_profiles_ui = []
        user_ids = list(
            recommendations_data.keys()
        )

        profiles_list = await client.get_profiles(
            user_ids
        )
        content_list = await client.get_profile_content(
            user_ids
        )

        profiles_map = {p.user_id: p for p in profiles_list}
        content_map = {c.user_id: c for c in content_list}

        print(
            f"Rendering {len(profiles_list)} profiles..."
        )

        for user_id, subject in recommendations_data.items():
            profile = profiles_map.get(
                user_id
            )
            content = content_map.get(
                user_id
            )

            if not profile or not content:
                continue

            profile_ui_elements = []
            header_text = f"<h2>{profile.profile.first_name}, {profile.profile.age} - {profile.profile.location.name}</h2>"
            profile_ui_elements.append(
                widgets.HTML(
                    header_text
                )
            )

            async def handle_action(
                action_coroutine,
                button
            ):
                button.disabled = True
                button.description = "Working..."
                try:
                    await action_coroutine
                    await update_like_limit_display()
                    button.description = "Done!"
                    button.button_style = 'success'
                except Exception as e:
                    button.description = "Error!"
                    button.button_style = 'warning'

            for photo in content.content.photos:
                comment_input = widgets.Text(
                    placeholder="Type a comment..."
                )
                like_button = widgets.Button(
                    description="❤️ Like Photo",
                    button_style='danger'
                )

                async def photo_like_handler(
                    p,
                    c,
                    b,
                    _
                ):
                    await handle_action(
                        p.like(
                            comment=c.value if c.value else None
                        ),
                        b
                    )

                like_button.on_click(
                    lambda
                        _,
                        p=photo,
                        c=comment_input,
                        b=like_button: loop.run_until_complete(
                        photo_like_handler(
                            p,
                            c,
                            b,
                            _
                        )
                    )
                )

                profile_ui_elements.append(
                    widgets.VBox(
                        [
                            widgets.HTML(
                                f'<img src="{photo.url}" width="400">'
                            ),
                            widgets.HBox(
                                [comment_input, like_button]
                            )
                        ]
                    )
                )

            for answer in content.content.answers:
                prompt_text = await client.get_prompt_text(
                    answer.question_id
                )
                comment_input = widgets.Text(
                    placeholder="Type a comment..."
                )
                like_button = widgets.Button(
                    description="❤️ Like Prompt",
                    button_style='danger'
                )

                async def answer_like_handler(
                    a,
                    c,
                    b,
                    _
                ):
                    await handle_action(
                        a.like(
                            comment=c.value if c.value else None
                        ),
                        b
                    )

                like_button.on_click(
                    lambda
                        _,
                        a=answer,
                        c=comment_input,
                        b=like_button: loop.run_until_complete(
                        answer_like_handler(
                            a,
                            c,
                            b,
                            _
                        )
                    )
                )

                profile_ui_elements.append(
                    widgets.VBox(
                        [
                            widgets.HTML(
                                f"<b>{prompt_text}</b><br><i>{answer.response}</i>"
                            ),
                            widgets.HBox(
                                [comment_input, like_button]
                            )
                        ]
                    )
                )

            skip_button = widgets.Button(
                description="Skip Profile",
                button_style='warning'
            )

            async def skip_handler(
                s,
                b,
                _
            ):
                await handle_action(
                    s.skip(),
                    b
                )

            skip_button.on_click(
                lambda
                    _,
                    s=subject,
                    b=skip_button: loop.run_until_complete(
                    skip_handler(
                        s,
                        b,
                        _
                    )
                )
            )
            profile_ui_elements.append(
                skip_button
            )

            profile_ui_elements.append(
                widgets.HTML(
                    "<hr style='border: 1px solid #ccc;'>"
                )
            )
            all_profiles_ui.extend(
                profile_ui_elements
            )

        display(
            widgets.VBox(
                all_profiles_ui
            )
        )


async def fetch_recommendations_action():
    """Fetch recommendations and trigger the display function."""
    global recommendations_data
    with main_output:
        clear_output(
            wait=True
        )
        print(
            "Fetching recommendations..."
        )
        await client.get_recommendations()
        recommendations_data = client.recommendations
        if not recommendations_data:
            print(
                "No new recommendations found. Try recycling profiles."
            )
        else:
            await display_profiles()


async def recycle_profiles_action():
    """Recycle profiles and then fetch them."""
    with main_output:
        clear_output(
            wait=True
        )
        print(
            "Recycling profiles..."
        )
        await client.repeat_profiles()
        print(
            "Recycling complete. Fetching new queue..."
        )
        await fetch_recommendations_action()


def on_fetch_button_clicked(
    b
):
    loop.run_until_complete(
        fetch_recommendations_action()
    )


def on_recycle_button_clicked(
    b
):
    loop.run_until_complete(
        recycle_profiles_action()
    )


fetch_button.on_click(
    on_fetch_button_clicked
)
recycle_button.on_click(
    on_recycle_button_clicked
)

if auth_success:
    loop.run_until_complete(
        update_like_limit_display()
    )

control_panel = widgets.HBox(
    [fetch_button, recycle_button, like_limit_label]
)
display(
    control_panel
)
display(
    main_output
)

if auth_success and client.recommendations:
    print(
        "Pre-loading recommendations from your session..."
    )
    recommendations_data = client.recommendations
    loop.run_until_complete(
        display_profiles()
    )

2025-09-03T20:28:37.412183Z [info     ] Fetching like limits for authenticated user filename=hinge_client.py func_name=get_like_limit identity_id=3660266827386193054 lineno=803


connect_tcp.started host='prod-api.hingeaws.net' port=443 local_address=None timeout=30.0 socket_options=None
connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x10f24f550>
start_tls.started ssl_context=<ssl.SSLContext object at 0x10f027920> server_hostname='prod-api.hingeaws.net' timeout=30.0
start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x10f26bc90>
send_request_headers.started request=<Request [b'GET']>
send_request_headers.complete
send_request_body.started request=<Request [b'GET']>
send_request_body.complete
receive_response_headers.started request=<Request [b'GET']>
receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json'), (b'Content-Length', b'106'), (b'Connection', b'keep-alive'), (b'Date', b'Wed, 03 Sep 2025 20:28:37 GMT'), (b'server', b'istio-envoy'), (b'x-auth-state-id', b'standard_1'), (b'x-auth-state-permissions-checksum', b'37bda291f9f62ae961bdd9ba266

Output()

Pre-loading recommendations from your session...
